# 04 — Sampling and bootstrapping

This notebook builds a small review batch: examples that should be prioritised for expert review.

The point is not just to ask "What is the score?" but "What should we annotate next?".

In [1]:
!pip -q install scikit-learn

In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_DIR = Path('/content/lrec2026_llm_annotator')
DATA_DIR = PROJECT_DIR / 'data' / 'sample'
OUTPUT_DIR = PROJECT_DIR / 'outputs'

validated = pd.read_json(OUTPUT_DIR / 'validated_predictions.jsonl', lines=True)
gold = pd.read_csv(DATA_DIR / 'toy_sentences.csv')
for col in ['tokens','gold_pos','gold_lemma','gold_features']:
    gold[col] = gold[col].apply(json.loads)

gold[['id','language','domain','text','split']]

,id,language,domain,text,split
0,grc_001,Ancient Greek,toy,λόγος ἐστὶ καλός .,fewshot
1,grc_002,Ancient Greek,toy,οἱ ἄνδρες γράφουσι .,eval
2,grc_003,Ancient Greek,toy_noisy,βασιλεὺς εἶπεν λόγον .,eval
3,xcl_001,Classical Armenian,toy,այր մի եկն .,fewshot
4,xcl_002,Classical Armenian,toy,թագաւորն գրեաց նամակ .,eval
5,xcl_003,Classical Armenian,toy_noisy,աշակերտք ընթերցան գիրք .,eval
6,oge_001,Old Georgian,toy,კაცი მოვიდა .,fewshot
7,oge_002,Old Georgian,toy,წიგნი კეთილი არს .,eval
8,oge_003,Old Georgian,toy_noisy,მოწაფენი წერენ წიგნსა .,eval
9,syr_001,Syriac,toy,ܓܒܪܐ ܐܬܐ .,fewshot


## Sentence-level signals

We combine several signals:

- validation failure
- low confidence
- zero-shot vs few-shot disagreement
- use of fallback/unknown labels
- diversity across the corpus
- random slice for coverage

In [3]:
def sentence_confidence_score(parsed):
    # Higher = more uncertain.
    if not isinstance(parsed, dict) or 'tokens' not in parsed:
        return 1.0
    mapping = {'high': 0.0, 'medium': 0.5, 'low': 1.0}
    vals = [mapping.get(tok.get('confidence'), 0.75) for tok in parsed.get('tokens', []) if isinstance(tok, dict)]
    return float(np.mean(vals)) if vals else 1.0

def pos_sequence(parsed):
    if not isinstance(parsed, dict) or 'tokens' not in parsed:
        return []
    return [tok.get('upos') for tok in parsed['tokens'] if isinstance(tok, dict)]

# Pivot zero/few predictions by sentence.
records = []
for _, rec in validated.iterrows():
    records.append({
        'sentence_id': rec['sentence_id'],
        'language': rec['language'],
        'mode': rec['mode'],
        'is_valid': bool(rec['is_valid']),
        'parsed': rec['parsed'],
        'uncertainty': sentence_confidence_score(rec['parsed']),
        'pos_sequence': pos_sequence(rec['parsed']),
        'validation_errors': rec['validation_errors']
    })
sig_long = pd.DataFrame(records)
sig = sig_long.pivot_table(index='sentence_id', columns='mode', values='uncertainty', aggfunc='first').reset_index()
sig.columns.name = None
sig = sig.rename(columns={'zero_shot': 'uncertainty_zero', 'few_shot': 'uncertainty_few'})
sig

,sentence_id,uncertainty_few,uncertainty_zero
0,grc_002,0.125,0.125
1,grc_003,0.000,0.000
2,oge_002,0.000,0.000
3,oge_003,0.000,0.000
4,syr_002,0.125,0.250
5,syr_003,0.125,0.250
6,xcl_002,0.000,0.000
7,xcl_003,0.000,0.000


In [4]:
# Disagreement score between zero-shot and few-shot POS sequences.
def disagreement_for_sentence(sid):
    sub = sig_long[sig_long['sentence_id'] == sid]
    seqs = {row['mode']: row['pos_sequence'] for _, row in sub.iterrows()}
    z, f = seqs.get('zero_shot', []), seqs.get('few_shot', [])
    if not z or not f or len(z) != len(f):
        return 1.0
    return float(np.mean([a != b for a, b in zip(z, f)]))

sig['disagreement'] = sig['sentence_id'].apply(disagreement_for_sentence)

valid_fail = sig_long.groupby('sentence_id')['is_valid'].apply(lambda x: 1.0 - float(all(x))).reset_index(name='validation_failure')
sig = sig.merge(valid_fail, on='sentence_id', how='left')
sig['uncertainty'] = sig[['uncertainty_zero','uncertainty_few']].mean(axis=1)
sig

,sentence_id,uncertainty_few,uncertainty_zero,disagreement,validation_failure,uncertainty
0,grc_002,0.125,0.125,0.25,1.0,0.1250
1,grc_003,0.000,0.000,0.00,0.0,0.0000
2,oge_002,0.000,0.000,0.00,0.0,0.0000
3,oge_003,0.000,0.000,0.00,0.0,0.0000
4,syr_002,0.125,0.250,0.00,0.0,0.1875
5,syr_003,0.125,0.250,0.50,0.0,0.1875
6,xcl_002,0.000,0.000,1.00,1.0,0.0000
7,xcl_003,0.000,0.000,0.00,0.0,0.0000


## Diversity score

Here we use a simple character n-gram TF-IDF representation of the sentence text. In a real project, you might use embeddings and cluster-based sampling.

In [5]:
meta = gold.rename(columns={'id': 'sentence_id'})[['sentence_id','language','domain','text','tokens']]
sig = sig.merge(meta, on='sentence_id', how='left')

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X = vectorizer.fit_transform(sig['text'].fillna(''))
sim = cosine_similarity(X)
# Diversity proxy: 1 - average similarity to all other examples.
if len(sig) > 1:
    avg_sim = (sim.sum(axis=1) - 1) / (len(sig) - 1)
else:
    avg_sim = np.zeros(len(sig))
sig['diversity'] = 1 - avg_sim
sig[['sentence_id','language','text','diversity']]

,sentence_id,language,text,diversity
0,grc_002,Ancient Greek,οἱ ἄνδρες γράφουσι .,0.986142
1,grc_003,Ancient Greek,βασιλεὺς εἶπεν λόγον .,0.986785
2,oge_002,Old Georgian,წიგნი კეთილი არს .,0.959820
3,oge_003,Old Georgian,მოწაფენი წერენ წიგნსა .,0.961335
4,syr_002,Syriac,ܡܠܟܐ ܟܬܒ ܐܓܪܬܐ .,0.957124
5,syr_003,Syriac,ܬܠܡܝܕܐ ܩܪܐ ܟܬܒܐ .,0.957788
6,xcl_002,Classical Armenian,թագաւորն գրեաց նամակ .,0.985336
7,xcl_003,Classical Armenian,աշակերտք ընթերցան գիրք .,0.986020


## Hybrid priority score

Weights are placeholders. The point is to make the selection rule explicit and auditable.

In [6]:
# Normalize helper.
def normalize(s):
    s = pd.Series(s).astype(float)
    if s.max() == s.min():
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

sig['priority'] = (
    0.35 * normalize(sig['uncertainty']) +
    0.30 * normalize(sig['disagreement']) +
    0.20 * normalize(sig['diversity']) +
    0.15 * normalize(sig['validation_failure'])
)

sig = sig.sort_values('priority', ascending=False)
sig[['sentence_id','language','domain','uncertainty','disagreement','diversity','validation_failure','priority']]

,sentence_id,language,domain,uncertainty,disagreement,diversity,validation_failure,priority
0,grc_002,Ancient Greek,toy,0.1250,0.25,0.986142,1.0,0.653994
6,xcl_002,Classical Armenian,toy,0.0000,1.00,0.985336,1.0,0.640230
5,syr_003,Syriac,toy_noisy,0.1875,0.50,0.957788,0.0,0.504474
4,syr_002,Syriac,toy,0.1875,0.00,0.957124,0.0,0.350000
1,grc_003,Ancient Greek,toy_noisy,0.0000,0.00,0.986785,0.0,0.200000
7,xcl_003,Classical Armenian,toy_noisy,0.0000,0.00,0.986020,0.0,0.194838
3,oge_003,Old Georgian,toy_noisy,0.0000,0.00,0.961335,0.0,0.028394
2,oge_002,Old Georgian,toy,0.0000,0.00,0.959820,0.0,0.018178


## Build a stratified review batch

We select high-priority cases while keeping language coverage. For a real project, also reserve some random examples.

In [7]:
K_PER_LANGUAGE = 1
review_batch = (
    sig.sort_values(['language','priority'], ascending=[True, False])
       .groupby('language', group_keys=False)
       .head(K_PER_LANGUAGE)
       .sort_values('priority', ascending=False)
       .copy()
)

review_batch['selection_reason'] = review_batch.apply(
    lambda r: f"priority={r['priority']:.2f}; uncertainty={r['uncertainty']:.2f}; disagreement={r['disagreement']:.2f}; validation_failure={r['validation_failure']:.0f}",
    axis=1
)

review_path = OUTPUT_DIR / 'review_batch.csv'
review_batch.to_csv(review_path, index=False, encoding='utf-8')
print('Wrote', review_path)
review_batch[['sentence_id','language','text','selection_reason']]

Wrote /content/lrec2026_llm_annotator/outputs/review_batch.csv


,sentence_id,language,text,selection_reason
0,grc_002,Ancient Greek,οἱ ἄνδρες γράφουσι .,priority=0.65; uncertainty=0.12; disagreement=...
6,xcl_002,Classical Armenian,թագաւորն գրեաց նամակ .,priority=0.64; uncertainty=0.00; disagreement=...
5,syr_003,Syriac,ܬܠܡܝܕܐ ܩܪܐ ܟܬܒܐ .,priority=0.50; uncertainty=0.19; disagreement=...
3,oge_003,Old Georgian,მოწაფენი წერენ წიგნსა .,priority=0.03; uncertainty=0.00; disagreement=...


## Simulating the bootstrapping update

After expert review, corrected examples can become:

- new gold examples;
- few-shot examples;
- error cases in the guideline;
- training data for a specialised model.

In [8]:
# Placeholder for expert corrections.
# In a real workshop, participants would fill these columns manually or via a spreadsheet.
expert_template = review_batch[['sentence_id','language','text','tokens','selection_reason']].copy()
expert_template['expert_checked'] = False
expert_template['expert_notes'] = ''
expert_template_path = OUTPUT_DIR / 'expert_review_template.csv'
expert_template.to_csv(expert_template_path, index=False, encoding='utf-8')
print('Wrote', expert_template_path)
expert_template

Wrote /content/lrec2026_llm_annotator/outputs/expert_review_template.csv


,sentence_id,language,text,tokens,selection_reason,expert_checked,expert_notes
0,grc_002,Ancient Greek,οἱ ἄνδρες γράφουσι .,"[οἱ, ἄνδρες, γράφουσι, .]",priority=0.65; uncertainty=0.12; disagreement=...,False,
6,xcl_002,Classical Armenian,թագաւորն գրեաց նամակ .,"[թագաւորն, գրեաց, նամակ, .]",priority=0.64; uncertainty=0.00; disagreement=...,False,
5,syr_003,Syriac,ܬܠܡܝܕܐ ܩܪܐ ܟܬܒܐ .,"[ܬܠܡܝܕܐ, ܩܪܐ, ܟܬܒܐ, .]",priority=0.50; uncertainty=0.19; disagreement=...,False,
3,oge_003,Old Georgian,მოწაფენი წერენ წიგნსა .,"[მოწაფენი, წერენ, წიგნსა, .]",priority=0.03; uncertainty=0.00; disagreement=...,False,


## Participant TODO

Open `review_batch.csv` or `expert_review_template.csv` and decide whether the selected examples look genuinely useful.

Would you change the weights? Would you add a random slice? Would you stratify by domain instead of language?

## End of notebooks

You now have a full minimal workflow:

1. create/load data;
2. prompt zero/few-shot;
3. parse and validate;
4. evaluate and analyse errors;
5. select examples for expert review.